# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026 - gzk9gr - Nathaniel Garcia**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [ ]:
count = df.duplicated().sum()
df = df.drop_duplicates().copy()
log('duplicates', 'Removed duplicates', count)

[duplicates] Removed duplicates (15 row(s))


### TODO 2 — clean `price` -> float

In [ ]:
df['price'] = df['price'].str.replace('$', '', regex=False).astype(float)
log('price', 'Converted to float', len(df))

[price] Converted to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [ ]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
invalid = df['qty'].isna() | (df['qty'] < 0)
log('qty', 'Removed missing and negative quantities', invalid.sum())
df = df.loc[~invalid].copy()

[qty] Removed missing and negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [ ]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

items = df['item'].map(ITEM_MAP)
log('item', 'Standardized item names', (df['item'] != items).sum())
df['item'] = items

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] Standardized item names (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [ ]:
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

categories = df['category'].map(CATEGORY_MAP)
log('category', 'Standardized labels; grouped Apparel with Merch', (df['category'] != categories).sum())
df['category'] = categories

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] Standardized labels; grouped Apparel with Merch (132 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [ ]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['Cheeseburger', 'Foam Finger', 'Rain Poncho']).all()
assert df['category'].isin(['Food', 'Merch', 'RainGear']).all()
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [ ]:
df['revenue'] = df['qty'] * df['price']
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print(revenue_by_category.to_string(float_format='${:,.2f}'.format))
print(f"Total: ${df['revenue'].sum():,.2f}")

category
Food       $1,656.00
Merch      $1,572.00
RainGear   $1,512.00
Total: $4,740.00


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [ ]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Removed duplicates,15
1,price,Converted to float,300
2,qty,Removed missing and negative quantities,25
3,item,Standardized item names,126
4,category,Standardized labels; grouped Apparel with Merch,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** Removing duplicates changed revenue the most. Revenue went from **$4,852.50 to $4,594.50**, a decrease of **$258.00**. These totals use numeric prices and exclude missing quantities.

**b)** I grouped Apparel with Merch because apparel is a type of merchandise. Someone else could reasonably keep them separate. That would split their category revenues into separate groups, but the overall revenue would still be **$4,740.00**. I combined them to make the category report simpler.